# 🥛 Dairy Sales Forecasting — Improved Model Training
### Prophet + SARIMAX + XGBoost | Festival-Aware | Multi-Product

---

## 🎯 Project Objectives (from your report)
1. Study **demand patterns** with seasonal & festival variations  
2. Develop a **festival-aware forecasting model** (ML + DL)  
3. **Integrate festival calendar** with historical demand data  
4. Build decision-support **outputs** for production planning  

---

## 📋 How to Run This Notebook
**Step 1 →** Run Cell 1 (install packages)  
**Step 2 →** Run Cell 2 (imports)  
**Step 3 →** Run Cell 3 (set your CSV file path)  
**Step 4 →** Run Cells 4–11 one by one  
**Step 5 →** Use Cell 12 to query any festival

> **Input file:** `final_sales_with_festivals.csv`  
> **Required columns:** `Date`, `Product`, `Quantity`, `Festival` (Festival can be blank for non-festival days)

---
## 📦 Cell 1 — Install Dependencies
Run this once. It installs Prophet, SARIMAX (statsmodels), XGBoost, and other helper libraries.

In [ ]:
%%capture
import subprocess, sys
pkgs = [
    'prophet',
    'statsmodels',
    'scikit-learn',
    'xgboost',
    'pandas',
    'numpy',
    'openpyxl',
    'matplotlib',
    'seaborn'
]
for pkg in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=False)

print('✅ All packages installed successfully')

---
## 📥 Cell 2 — Imports
Importing all libraries needed across the notebook.  
✅ **Fix:** Removed `google.colab` dependency — this notebook now works in **Jupyter, VS Code, and Google Colab**.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import pickle, os, json
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Models
from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from xgboost import XGBRegressor

# Metrics
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# Create output folders
os.makedirs('saved_models/prophet',  exist_ok=True)
os.makedirs('saved_models/sarimax',  exist_ok=True)
os.makedirs('saved_models/xgboost',  exist_ok=True)
os.makedirs('outputs',               exist_ok=True)

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'Prophet': '#4C72B0', 'SARIMAX': '#DD8452', 'XGBoost': '#55A868'}

print('✅ All imports successful')

---
## 📂 Cell 3 — Load Data
**Set your CSV file path below.**  
Works both in Colab (using file upload) and in local Jupyter (using a file path).

In [ ]:
# ─────────────────────────────────────────────────────────────
# ★ SET YOUR FILE PATH HERE
# ─────────────────────────────────────────────────────────────
CSV_PATH = 'final_sales_with_festivals.csv'   # ← change this if your file is elsewhere

# Auto-detect Colab and use file upload if running there
try:
    import google.colab
    from google.colab import files
    print('Running in Google Colab — please upload your CSV:')
    uploaded = files.upload()
    CSV_PATH = list(uploaded.keys())[0]
    print(f'Uploaded: {CSV_PATH}')
except ImportError:
    print(f'Running locally — loading: {CSV_PATH}')

# ── CONFIG ──
TEST_DAYS  = 90    # last N days used for testing (not training)
MIN_ROWS   = 60    # products with fewer than this many rows are skipped
FEST_WINDOW = 2    # festival effect window: N days before and after

# ── Load ──
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()

print(f'\nColumns found  : {df.columns.tolist()}')
print(f'Shape          : {df.shape}')
df.head(5)

---
## 🧹 Cell 4 — Clean & Validate Data
**What happens here:**
- Rename columns to standard names (`ds`, `y`, `product`, `festival`)
- Parse dates correctly
- Remove rows with missing quantity or date
- Compute baseline sales (average on non-festival days) per product

**Why baselines matter:** They help us calculate how much MORE is sold on festival days vs normal days — this is the "spike %".

In [ ]:
# ── Rename to standard names ──
df = df.rename(columns={
    'Date':     'ds',
    'Quantity': 'y',
    'Product':  'product',
    'Festival': 'festival'
})

# ── Parse dates (handles both DD/MM/YYYY and MM/DD/YYYY) ──
df['ds'] = pd.to_datetime(df['ds'], errors='coerce', dayfirst=True)
bad_dates = df['ds'].isna().sum()
if bad_dates > 0:
    print(f'⚠  Dropped {bad_dates} rows with unparseable dates')

# ── Clean other columns ──
df['festival'] = df['festival'].fillna('').astype(str).str.strip()
df['y']        = pd.to_numeric(df['y'], errors='coerce')
df = df.dropna(subset=['y', 'ds']).sort_values('ds').reset_index(drop=True)

# ── Filter to products with enough data ──
prod_counts = df['product'].value_counts()
products    = prod_counts[prod_counts >= MIN_ROWS].index.tolist()

print('=' * 55)
print(f'  Total rows          : {len(df):,}')
print(f'  Total products      : {df["product"].nunique()}')
print(f'  Trainable (≥{MIN_ROWS} rows): {len(products)}')
print(f'  Date range          : {df["ds"].min().date()} → {df["ds"].max().date()}')
print(f'  Unique festivals    : {df[df["festival"] != ""]["festival"].nunique()}')
print('=' * 55)

# ── OBJECTIVE 1: Compute baselines (normal day average per product) ──
# Baseline = average quantity sold on days with NO festival
non_fest  = df[df['festival'] == '']
baselines = (
    non_fest.groupby('product')['y']
    .agg(baseline_mean='mean', baseline_std='std')
    .round(2)
)
baselines.to_csv('outputs/product_baselines.csv')
print('\n✅ Product baselines saved (outputs/product_baselines.csv)')
print('   These are the NORMAL day averages — used to calculate festival spike %')
baselines.head()

---
## 📊 Cell 5 — Explore Demand Patterns (Objective 1)
**Objective 1:** Understand demand patterns w.r.t. seasonal and festival variations.

This cell generates charts to visualize:
- Monthly seasonal trends
- Festival vs Non-festival demand comparison

In [ ]:
# ── Monthly trend for top 3 products ──
df['month'] = df['ds'].dt.to_period('M')
top3 = prod_counts.head(3).index.tolist()

fig, axes = plt.subplots(len(top3), 1, figsize=(14, 4 * len(top3)))
if len(top3) == 1:
    axes = [axes]

for ax, prod in zip(axes, top3):
    sub = df[df['product'] == prod].copy()
    monthly = sub.groupby('month')['y'].sum().reset_index()
    monthly['month_dt'] = monthly['month'].dt.to_timestamp()
    ax.plot(monthly['month_dt'], monthly['y'], marker='o', linewidth=2, color='#4C72B0')
    
    # Mark festival months
    fest_months = sub[sub['festival'] != ''].groupby('month')['y'].sum()
    if not fest_months.empty:
        fm_dt = fest_months.index.to_timestamp()
        ax.scatter(fm_dt, fest_months.values, color='red', zorder=5, s=80, label='Festival month')
    
    ax.set_title(f'Monthly Sales — {prod}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Total Quantity')
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.suptitle('Objective 1: Seasonal Demand Patterns (Red = Festival Months)', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outputs/seasonal_trends.png', bbox_inches='tight', dpi=100)
plt.show()

# ── Festival vs Non-festival comparison ──
df['is_festival'] = df['festival'].apply(lambda x: 'Festival Day' if x != '' else 'Normal Day')
fig2, ax2 = plt.subplots(figsize=(10, 5))
comparison = df.groupby(['product', 'is_festival'])['y'].mean().reset_index()
top5 = prod_counts.head(5).index.tolist()
comp_top5 = comparison[comparison['product'].isin(top5)]

comp_pivot = comp_top5.pivot(index='product', columns='is_festival', values='y')
comp_pivot.plot(kind='bar', ax=ax2, color=['#4C72B0', '#DD8452'], edgecolor='white')
ax2.set_title('Average Daily Quantity: Festival Days vs Normal Days (Top 5 Products)', fontsize=12)
ax2.set_xlabel('')
ax2.set_ylabel('Average Quantity')
ax2.tick_params(axis='x', rotation=30)
ax2.legend(title='')
plt.tight_layout()
plt.savefig('outputs/festival_vs_normal.png', bbox_inches='tight', dpi=100)
plt.show()

print('✅ Charts saved to outputs/')

---
## 📅 Cell 6 — Build Festival Calendar (Objective 3)
**Objective 3:** Integrate traditional festival calendar with historical demand data.

**What this cell does:**
- Extracts all festival dates from your CSV
- Creates a **Prophet holiday dataframe** (models know which days are festivals)
- Creates a **SARIMAX exogenous matrix** — a binary (0/1) table where each column is a festival and each row is a date (1 = festival window active that day)
- Creates **XGBoost festival features** for the feature-engineering approach

**Festival window:** Each festival's effect is assumed to start `FEST_WINDOW` days before and end `FEST_WINDOW` days after the actual date.

In [ ]:
# ── Extract unique festival → date mapping ──
fest_raw = (
    df[df['festival'] != ''][['ds', 'festival']]
    .drop_duplicates()
    .copy()
)

# ── Prophet holiday dataframe ──
# Prophet needs: columns 'ds' (date), 'holiday' (name), 'lower_window', 'upper_window'
fest_df = fest_raw.rename(columns={'festival': 'holiday'}).copy()
fest_df['lower_window'] = -FEST_WINDOW   # effect starts N days BEFORE the festival
fest_df['upper_window'] =  FEST_WINDOW   # effect ends N days AFTER the festival
fest_df = fest_df.reset_index(drop=True)

known_festivals = sorted(fest_df['holiday'].unique().tolist())

# Save festival list
with open('outputs/known_festivals.json', 'w') as f:
    json.dump(known_festivals, f, indent=2)

# ── SARIMAX exogenous matrix ──
# Rows = all dates in dataset, Columns = each festival
# Value = 1 if that date falls within the festival window, else 0
all_dates = pd.date_range(df['ds'].min(), df['ds'].max(), freq='D')
exog_all  = pd.DataFrame(0, index=all_dates, columns=known_festivals)
exog_all.index.name = 'ds'

for fest in known_festivals:
    fest_dates = fest_raw[fest_raw['festival'] == fest]['ds'].values
    for fd in fest_dates:
        mask = (all_dates >= fd - pd.Timedelta(days=FEST_WINDOW)) & \
               (all_dates <= fd + pd.Timedelta(days=FEST_WINDOW))
        exog_all.loc[mask, fest] = 1

exog_all.to_csv('outputs/festival_exog_matrix.csv')

# ── Festival statistics table ──
# Shows: how many times each festival appears, average demand boost
fest_stats = []
for fest in known_festivals:
    fest_days = df[df['festival'] == fest]
    if len(fest_days) == 0:
        continue
    for prod in fest_days['product'].unique():
        prod_fest  = fest_days[fest_days['product'] == prod]['y'].mean()
        prod_base  = baselines.loc[prod, 'baseline_mean'] if prod in baselines.index else None
        spike_pct  = ((prod_fest - prod_base) / prod_base * 100) if prod_base else None
        fest_stats.append({'Festival': fest, 'Product': prod,
                           'Avg_Demand_On_Festival': round(prod_fest, 2),
                           'Baseline_Demand': round(prod_base, 2) if prod_base else None,
                           'Spike_%': round(spike_pct, 2) if spike_pct else None})

fest_stats_df = pd.DataFrame(fest_stats)
fest_stats_df.to_csv('outputs/festival_demand_stats.csv', index=False)

print('Festival Calendar built:')
print(f'  Known festivals   : {len(known_festivals)}')
print(f'  Festival window   : ±{FEST_WINDOW} days around each festival')
print(f'  Prophet holiday df: {len(fest_df)} rows')
print(f'  SARIMAX exog shape: {exog_all.shape}')
print(f'\n  Festivals in your data:')
for i, f in enumerate(known_festivals, 1):
    count = len(fest_raw[fest_raw['festival'] == f])
    print(f'    {i:2}. {f} ({count} date(s) in dataset)')
print('\n✅ Festival calendar ready (Objective 3 complete)')

---
## 📆 Cell 7 — Year-Wise Festival Spike Analysis (NEW — Production Planning)
**What this cell answers:**
> *"For each product and each festival, how much did demand spike in 2019, 2020, 2021… and is the spike growing or shrinking year by year?"*

**Why this matters for the dairy production team:**
- If Diwali spike for Shrikhand was 40% in 2020 → 55% in 2021 → 70% in 2022, you should plan for ~85% in 2023
- If a product's spike is **declining**, you don't need to over-produce
- This gives the production manager a single table to plan stock for every festival in the coming year

**Outputs:**
- `outputs/yearwise_spike_analysis.csv` — master table (Product × Festival × Year → Spike%)
- `outputs/yearwise_next_year_forecast.csv` — predicted spike % for next year (trend extrapolation)
- Heatmap charts per festival showing spike % across products and years
- Trend chart per product showing if spike is growing/shrinking year by year

In [ ]:
# ══════════════════════════════════════════════════════════════════
# YEAR-WISE SPIKE ANALYSIS
# For each (Product, Festival, Year): compute actual avg demand
# vs that year's baseline → spike %
# ══════════════════════════════════════════════════════════════════

import matplotlib.ticker as mticker
from scipy import stats as scipy_stats

df['year'] = df['ds'].dt.year
years      = sorted(df['year'].unique())
print(f'Years in dataset: {years}')
print(f'Products to analyse: {len(products)}')
print(f'Festivals to analyse: {len(known_festivals)}')
print()

# ── Step 1: Per-year, per-product baseline (normal day average) ──
# We compute a SEPARATE baseline for each year so year-on-year growth
# doesn't inflate or deflate the spike calculation
non_fest_rows = df[df['festival'] == '']
yearwise_baselines = (
    non_fest_rows
    .groupby(['year', 'product'])['y']
    .agg(baseline_mean='mean', baseline_std='std', baseline_days='count')
    .reset_index()
)
yearwise_baselines.to_csv('outputs/yearwise_baselines.csv', index=False)
print('✅ Year-wise baselines computed')
print(yearwise_baselines.head(6).to_string(index=False))

# ── Step 2: Per-year, per-product, per-festival actual demand ──
fest_rows = df[df['festival'] != ''].copy()
yearwise_fest_demand = (
    fest_rows
    .groupby(['year', 'product', 'festival'])['y']
    .agg(fest_mean='mean', fest_std='std', fest_days='count')
    .reset_index()
)

# ── Step 3: Merge baseline into festival rows → compute spike % ──
spike_df = yearwise_fest_demand.merge(
    yearwise_baselines[['year','product','baseline_mean','baseline_std']],
    on=['year','product'],
    how='left'
)

# Spike % = (festival_avg - baseline_avg) / baseline_avg × 100
spike_df['spike_pct'] = (
    (spike_df['fest_mean'] - spike_df['baseline_mean'])
    / spike_df['baseline_mean'] * 100
).round(2)

# Extra units per day = fest_mean - baseline_mean
spike_df['extra_units_per_day'] = (spike_df['fest_mean'] - spike_df['baseline_mean']).round(1)

# Total extra units for that festival occurrence
spike_df['total_extra_units']   = (spike_df['extra_units_per_day'] * spike_df['fest_days']).round(1)

# Confidence: A = enough data, B = limited, C = very few days
spike_df['data_confidence'] = spike_df['fest_days'].apply(
    lambda x: 'A (good)' if x >= 5 else ('B (limited)' if x >= 2 else 'C (1 day only)')
)

# Sort for readability
spike_df = spike_df.sort_values(['festival','product','year']).reset_index(drop=True)
spike_df.to_csv('outputs/yearwise_spike_analysis.csv', index=False)

print('\n✅ Year-wise spike analysis saved → outputs/yearwise_spike_analysis.csv')
print(f'   Total rows: {len(spike_df)}')
print(f'   Columns: {spike_df.columns.tolist()}')
print('\nSample rows:')
spike_df[['year','product','festival','fest_mean','baseline_mean',
           'spike_pct','extra_units_per_day','data_confidence']].head(10)

### 📊 Cell 7b — Year-Wise Heatmaps (Festival × Year per Product)
Each heatmap = one festival.  
Rows = products, Columns = years.  
Color = spike % (green = high spike, white = no change, red = demand dropped).

In [ ]:
# ── Heatmap: for each festival → Product (rows) × Year (cols) → Spike % ──
os.makedirs('outputs/yearwise_heatmaps', exist_ok=True)

for fest in known_festivals:
    sub = spike_df[spike_df['festival'] == fest]
    if sub.empty:
        continue

    # Pivot: rows=product, cols=year, values=spike_pct
    pivot = sub.pivot_table(index='product', columns='year',
                            values='spike_pct', aggfunc='mean')
    if pivot.empty or pivot.shape[1] < 1:
        continue

    # Limit to top 15 products by avg spike for readability
    pivot['avg'] = pivot.mean(axis=1)
    pivot = pivot.sort_values('avg', ascending=False).head(15).drop(columns='avg')

    fig_h = max(4, len(pivot) * 0.55)
    fig_w = max(6, len(pivot.columns) * 1.2)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn',
                   vmin=-20, vmax=max(60, pivot.values.max() if not np.isnan(pivot.values).all() else 60))

    # Axis labels
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([str(y) for y in pivot.columns], fontsize=10)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([p[:35] for p in pivot.index], fontsize=9)

    # Annotate each cell with the spike % value
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                color = 'white' if abs(val) > 40 else 'black'
                ax.text(j, i, f'{val:.1f}%', ha='center', va='center',
                        fontsize=8, color=color, fontweight='bold')
            else:
                ax.text(j, i, 'N/A', ha='center', va='center',
                        fontsize=7, color='gray')

    cbar = plt.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label('Spike % vs normal day baseline', fontsize=9)

    ax.set_title(f'{fest} — Demand Spike % by Product & Year\n'
                 f'(Green = high spike → produce MORE | Red = demand dropped → produce LESS)',
                 fontsize=11, fontweight='bold', pad=14)
    ax.set_xlabel('Year', fontsize=10)
    ax.set_ylabel('Product', fontsize=10)

    plt.tight_layout()
    safe_name = fest.replace(' ', '_').replace('/', '_')
    plt.savefig(f'outputs/yearwise_heatmaps/{safe_name}_heatmap.png',
                bbox_inches='tight', dpi=120)
    plt.show()
    print(f'  Saved heatmap → outputs/yearwise_heatmaps/{safe_name}_heatmap.png')

print('\n✅ All festival heatmaps saved')

### 📈 Cell 7c — Year-on-Year Spike Trend + Next-Year Forecast
**This is the key output for the production team.**

For each (Product × Festival), we:
1. Plot the spike % for each year as a line
2. Fit a linear trend to detect if it's growing, stable, or shrinking
3. **Extrapolate to next year** using that trend
4. Save a single master table `yearwise_next_year_forecast.csv` that the production manager can open in Excel

**How to read the trend direction:**
- ↑ Growing — plan to produce more than last year's spike
- → Stable — plan same as last year's spike
- ↓ Declining — last year's spike may have been a one-off, plan conservatively

In [ ]:
# ══════════════════════════════════════════════════════════════════
# For each product × festival: fit linear trend over years
# and project spike % for next year
# ══════════════════════════════════════════════════════════════════

next_year        = max(years) + 1
forecast_rows    = []
os.makedirs('outputs/yearwise_trends', exist_ok=True)

# We'll also make a combined multi-product trend chart per festival
for fest in known_festivals:
    sub_fest = spike_df[spike_df['festival'] == fest].copy()
    if sub_fest.empty:
        continue

    prods_in_fest = sub_fest['product'].unique()

    # ── Trend + forecast per product within this festival ──
    for product in prods_in_fest:
        sub = sub_fest[sub_fest['product'] == product].sort_values('year')
        yr_vals  = sub['year'].values
        spk_vals = sub['spike_pct'].values

        # Need at least 2 data points for a trend
        if len(yr_vals) < 2:
            slope, intercept, r2 = 0, float(spk_vals[0]) if len(spk_vals) else 0, None
            trend_dir = '→ Only 1 year of data'
        else:
            slope, intercept, r, p, _ = scipy_stats.linregress(yr_vals, spk_vals)
            r2 = round(r**2, 3)
            if   slope >  2.0: trend_dir = '↑ Growing'
            elif slope < -2.0: trend_dir = '↓ Declining'
            else:              trend_dir = '→ Stable'

        predicted_spike = round(intercept + slope * next_year, 2)
        # Cap prediction at reasonable bounds
        predicted_spike = max(-50, min(300, predicted_spike))

        last_year_spike   = float(spk_vals[-1])
        last_baseline     = float(sub.iloc[-1]['baseline_mean'])
        extra_units_nextyr = round(last_baseline * predicted_spike / 100, 1)
        plan_qty_nextyr    = round(last_baseline * (1 + predicted_spike / 100), 1)

        forecast_rows.append({
            'Festival':              fest,
            'Product':               product,
            'Years_Observed':        list(yr_vals.astype(int)),
            'Spike_%_Per_Year':      list(spk_vals.round(1)),
            'Trend_Direction':       trend_dir,
            'Trend_Slope_per_Year':  round(float(slope), 2),
            'R2_Fit_Quality':        r2,
            'Last_Year_Spike_%':     round(last_year_spike, 2),
            'Predicted_Spike_%_Next_Year': predicted_spike,
            'Baseline_Daily_Qty':    round(last_baseline, 1),
            'Extra_Units_Per_Day_NextYear': extra_units_nextyr,
            'Recommended_Daily_Qty_NextYear': plan_qty_nextyr,
            'Recommendation':        (
                f'Produce {plan_qty_nextyr:.0f} units/day during {fest} window'
                f' ({predicted_spike:+.1f}% vs normal {last_baseline:.0f} units/day)'
            )
        })

    # ── Combined trend chart for this festival (top 8 products) ──
    sub_top = sub_fest.copy()
    top_prods = (
        sub_top.groupby('product')['spike_pct'].mean()
        .sort_values(ascending=False).head(8).index.tolist()
    )
    sub_top = sub_top[sub_top['product'].isin(top_prods)]

    if sub_top.empty:
        continue

    fig, ax = plt.subplots(figsize=(10, 5))
    cmap_lines = plt.cm.get_cmap('tab10', len(top_prods))

    for idx, product in enumerate(top_prods):
        sub_p = sub_top[sub_top['product'] == product].sort_values('year')
        if sub_p.empty:
            continue
        yr_v  = sub_p['year'].values
        spk_v = sub_p['spike_pct'].values
        color = cmap_lines(idx)

        # Plot actual data points
        ax.plot(yr_v, spk_v, marker='o', linewidth=2.2, color=color,
                label=product[:30], zorder=3)

        # Draw trend line and project to next year
        if len(yr_v) >= 2:
            slope, intercept, *_ = scipy_stats.linregress(yr_v, spk_v)
            trend_x   = np.array([yr_v[0], next_year])
            trend_y   = intercept + slope * trend_x
            ax.plot(trend_x, trend_y, linestyle='--', linewidth=1.2,
                    color=color, alpha=0.55)
            # Mark projected point
            proj_y = round(intercept + slope * next_year, 1)
            ax.scatter([next_year], [proj_y], marker='*', s=120,
                       color=color, zorder=5)
            ax.annotate(f'{proj_y:.1f}%',
                        xy=(next_year, proj_y),
                        xytext=(5, 0), textcoords='offset points',
                        fontsize=7.5, color=color)

    ax.axhline(0, color='black', linewidth=0.8, linestyle='-', alpha=0.4)
    ax.axvline(max(years) + 0.5, color='gray', linewidth=1, linestyle=':',
               alpha=0.6, label=f'{next_year} forecast →')
    ax.set_xticks(list(years) + [next_year])
    ax.set_xticklabels([str(y) for y in years] + [f'{next_year}\n(forecast)'],
                       fontsize=9)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.set_title(f'{fest} — Year-on-Year Demand Spike Trend per Product\n'
                 f'(★ = predicted spike for {next_year} | dashed line = trend)',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Spike % vs Normal Day Baseline')
    ax.legend(loc='upper left', fontsize=8, framealpha=0.85,
              title='Products (top 8 by avg spike)')
    plt.tight_layout()
    safe_name = fest.replace(' ', '_').replace('/', '_')
    plt.savefig(f'outputs/yearwise_trends/{safe_name}_trend.png',
                bbox_inches='tight', dpi=120)
    plt.show()
    print(f'  Saved trend chart → outputs/yearwise_trends/{safe_name}_trend.png')

# ── Save master forecast table ──
forecast_df = pd.DataFrame(forecast_rows)
# Flatten list columns for CSV
forecast_csv = forecast_df.copy()
forecast_csv['Years_Observed']    = forecast_csv['Years_Observed'].apply(lambda x: ', '.join(map(str, x)))
forecast_csv['Spike_%_Per_Year']  = forecast_csv['Spike_%_Per_Year'].apply(lambda x: ', '.join(map(str, x)))
forecast_csv = forecast_csv.sort_values(['Festival','Predicted_Spike_%_Next_Year'],
                                         ascending=[True, False]).reset_index(drop=True)
forecast_csv.to_csv('outputs/yearwise_next_year_forecast.csv', index=False)

print(f'\n✅ Next-year forecast saved → outputs/yearwise_next_year_forecast.csv')
print(f'   Rows: {len(forecast_csv)} (Product × Festival combinations)')
print()

# ── Print top recommendations ──
print('═' * 75)
print('  TOP PRODUCTION RECOMMENDATIONS FOR', next_year)
print('  (Products with highest predicted spike % — sort by festival)')
print('═' * 75)
display_cols = ['Festival','Product','Last_Year_Spike_%',
                'Predicted_Spike_%_Next_Year','Trend_Direction',
                'Recommended_Daily_Qty_NextYear']
top_recs = forecast_csv.nlargest(20, 'Predicted_Spike_%_Next_Year')[display_cols]
print(top_recs.to_string(index=False))
print()
print('Interpretation guide:')
print('  ↑ Growing  → plan MORE than last year')
print('  → Stable   → plan SAME as last year')
print('  ↓ Declining→ plan LESS than last year')
forecast_csv[display_cols].head(20)

### 📋 Cell 7d — Production Planning Summary Table
Final clean table the dairy production manager can use directly.  
One row per (Festival × Product) sorted by festival date order.  
Shows: normal production qty → festival production qty → how many extra units to prepare.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PRODUCTION PLANNING SUMMARY
# Clean, manager-friendly table
# ══════════════════════════════════════════════════════════════════

print('DAIRY PRODUCTION PLANNING SUMMARY')
print(f'Forecast Year: {next_year}')
print('Based on year-wise historical spike trend analysis')
print('=' * 90)

# Group by festival for a clean layout
for fest in known_festivals:
    fest_recs = forecast_csv[forecast_csv['Festival'] == fest].copy()
    if fest_recs.empty:
        continue
    # Only show products with a positive spike prediction
    fest_recs = fest_recs[fest_recs['Predicted_Spike_%_Next_Year'] > 0]
    fest_recs = fest_recs.sort_values('Predicted_Spike_%_Next_Year', ascending=False)

    if fest_recs.empty:
        continue

    print(f'\n🎉 {fest}')
    print(f'  {"Product":<38} {"Normal Qty":>10} {"Fest Qty":>9} {"Extra/Day":>10} {"Spike%":>8} {"Trend":>12}')
    print('  ' + '-'*88)

    for _, row in fest_recs.iterrows():
        print(
            f"  {row['Product'][:38]:<38}"
            f"  {row['Baseline_Daily_Qty']:>8.1f}"
            f"  {row['Recommended_Daily_Qty_NextYear']:>8.1f}"
            f"  {row['Extra_Units_Per_Day_NextYear']:>9.1f}"
            f"  {row['Predicted_Spike_%_Next_Year']:>7.1f}%"
            f"  {row['Trend_Direction']:>12}"
        )

print('\n' + '=' * 90)
print('NOTE: Festival window = ±', FEST_WINDOW, 'days around the festival date')
print('      "Fest Qty" = recommended daily production during the festival window')
print('      "Extra/Day" = extra units to produce on top of normal production')
print('      Trend direction based on linear regression of historical spike % values')
print('\n✅ Save outputs/yearwise_next_year_forecast.csv to Excel for the production team')

---
## 🔮 Cell 8 — Train Prophet Model (Objective 2)
**What is Prophet?**  
Prophet is a time-series model by Meta (Facebook). It decomposes sales into:
- **Trend** — long-term direction (going up or down over months/years)
- **Yearly seasonality** — same pattern repeats every year (e.g., summer spike)
- **Weekly seasonality** — pattern repeats every week (e.g., weekends sell more)
- **Holiday effects** — how much extra (or less) is sold during festivals

**Multiplicative mode** means: if the baseline is 100 units and the holiday effect is +0.25, Prophet predicts 125 units (25% more).  

**Festival spike extraction:**  
After training, we read the `holidays` column from Prophet's forecast — this directly tells us the % lift for each festival.

In [ ]:
prophet_metrics   = []
prophet_spikes    = {}   # {product: {festival: spike_%}}
prophet_forecasts = {}   # {product: forecast_dataframe}

print(f'Training {len(products)} Prophet models...\n')
print(f'{"Product":<40} {"MAE":>8} {"MAPE%":>8} {"RMSE":>8} {"Grade":>6}')
print('-' * 75)

for product in products:
    df_p = df[df['product'] == product][['ds', 'y']].dropna().copy()

    # Split: train = all data except last TEST_DAYS days
    data_end = df_p['ds'].max()
    cutoff   = data_end - pd.Timedelta(days=TEST_DAYS)
    train_df = df_p[df_p['ds'] <= cutoff]
    test_df  = df_p[df_p['ds']  > cutoff]

    if len(train_df) < 30 or len(test_df) == 0:
        print(f'  {product[:40]:<40} SKIPPED (train={len(train_df)}, test={len(test_df)})')
        continue

    try:
        # Build Prophet model with Indian festivals
        m = Prophet(
            holidays           = fest_df,          # our festival calendar
            yearly_seasonality = True,             # detect annual patterns
            weekly_seasonality = True,             # detect weekly patterns
            daily_seasonality  = False,            # not needed for daily data
            seasonality_mode   = 'multiplicative', # better for data that scales with trend
            interval_width     = 0.90,             # 90% confidence intervals
            changepoint_prior_scale = 0.05,        # controls how flexible the trend is
        )
        m.add_country_holidays(country_name='IN')  # adds standard Indian holidays
        m.fit(train_df)

        # Forecast for the test period
        future   = m.make_future_dataframe(periods=TEST_DAYS)
        forecast = m.predict(future)
        prophet_forecasts[product] = forecast

        # Match forecast dates to actual test dates
        test_fc = forecast[forecast['ds'].isin(test_df['ds'])][['ds', 'yhat']]
        merged  = test_df.merge(test_fc, on='ds')
        if len(merged) == 0:
            merged = test_df.copy()
            merged['yhat'] = forecast.iloc[-len(test_df):]['yhat'].values

        # Calculate error metrics
        mae  = mean_absolute_error(merged['y'], merged['yhat'])
        mape = mean_absolute_percentage_error(merged['y'], merged['yhat']) * 100
        rmse = float(np.sqrt(((merged['y'] - merged['yhat']) ** 2).mean()))
        # Grade: A = very accurate (<15% MAPE), B = decent (<25%), C = needs work
        grade = 'A' if mape < 15 else ('B' if mape < 25 else 'C')

        # Extract festival spike effect from Prophet's forecast
        # 'holidays' column = fractional lift in multiplicative mode
        # e.g., 0.30 → 30% above baseline on that festival day
        spike_map = {}
        if 'holidays' in forecast.columns:
            holiday_fc = forecast.merge(
                fest_df[['ds', 'holiday']].drop_duplicates(), on='ds', how='left'
            )
            for fest_name, grp in holiday_fc.dropna(subset=['holiday']).groupby('holiday'):
                spike_map[fest_name] = round(float(grp['holidays'].mean()) * 100, 2)
        prophet_spikes[product] = spike_map

        # Save model to disk
        safe = product.replace('/', '_').replace(' ', '_')[:40]
        with open(f'saved_models/prophet/{safe}.pkl', 'wb') as fh:
            pickle.dump({'model': m, 'train': train_df, 'test': test_df,
                         'forecast': forecast}, fh)

        prophet_metrics.append({
            'Product':    product,
            'Train_Rows': len(train_df),
            'Test_Rows':  len(test_df),
            'MAE':        round(mae, 2),
            'RMSE':       round(rmse, 2),
            'MAPE_%':     round(mape, 2),
            'Grade':      grade
        })
        print(f'  {product[:40]:<40} {mae:>8.1f} {mape:>7.1f}% {rmse:>8.1f}  {grade}')

    except Exception as e:
        print(f'  {product[:40]:<40} FAILED: {e}')

# Save outputs
prophet_metrics_df = pd.DataFrame(prophet_metrics).sort_values('MAPE_%')
prophet_metrics_df.to_csv('outputs/prophet_metrics.csv', index=False)

prophet_spike_df = pd.DataFrame(prophet_spikes).T
prophet_spike_df.index.name = 'Product'
prophet_spike_df.to_csv('outputs/prophet_festival_spikes.csv')

print(f'\n✅ Prophet training complete: {len(prophet_metrics)} models')
print(f'   Average MAPE: {prophet_metrics_df["MAPE_%"].mean():.1f}%')
print(f'   Grade A (MAPE<15%): {(prophet_metrics_df["Grade"]=="A").sum()} products')
prophet_metrics_df

---
## 📈 Cell 9 — Train SARIMAX Model (Objective 2)
**What is SARIMAX?**  
SARIMAX = **S**easonal **A**uto**R**egressive **I**ntegrated **M**oving **A**verage with e**X**ogenous variables.

- **AR (AutoRegressive):** today's sales depend on past sales
- **MA (Moving Average):** smooths out random noise using past errors
- **I (Integrated):** differences the data to remove trend (make it stationary)
- **Seasonal:** same patterns repeat every 7 days (weekly season)
- **X (Exogenous):** uses festival dummies (0/1) as extra inputs → gives explicit **festival coefficients**

**Festival coefficient meaning:**  
If the Diwali coefficient for Product X is `45.2`, it means:
> "On Diwali (and ±2 days around it), we expect 45.2 extra units sold per day."

In [ ]:
# SARIMAX order: (p, d, q) × (P, D, Q, s)
# p=1: use yesterday's value, d=auto (stationarity test), q=1: use yesterday's error
# Seasonal s=7: weekly seasonality (7-day cycle)
SARIMAX_ORDER    = (1, 1, 1)
SARIMAX_SEASONAL = (1, 1, 1, 7)

def get_d(series):
    """
    ADF (Augmented Dickey-Fuller) test decides if we need differencing.
    - p-value < 0.05 → data is already stationary → d=0
    - p-value ≥ 0.05 → data has trend → d=1 (take 1st difference)
    """
    try:
        p_value = adfuller(series.dropna())[1]
        return 0 if p_value < 0.05 else 1
    except:
        return 1   # default to differencing if test fails

sarimax_metrics = []
sarimax_coeffs  = {}   # {product: {festival: coefficient}}

print(f'Training {len(products)} SARIMAX models...\n')
print(f'{"Product":<40} {"MAE":>8} {"MAPE%":>8} {"RMSE":>8} {"d":>4} {"Grade":>6}')
print('-' * 78)

for product in products:
    # Set daily frequency and forward-fill any gaps
    df_p = (
        df[df['product'] == product][['ds', 'y']]
        .dropna()
        .set_index('ds')
        .asfreq('D')
        .fillna(method='ffill')
    )

    cutoff    = df_p.index.max() - pd.Timedelta(days=TEST_DAYS)
    train_ser = df_p[df_p.index <= cutoff]['y']
    test_ser  = df_p[df_p.index  > cutoff]['y']

    if len(train_ser) < 30 or len(test_ser) == 0:
        print(f'  {product[:40]:<40} SKIPPED (train={len(train_ser)}, test={len(test_ser)})')
        continue

    # Align exogenous festival matrix to this product's date range
    train_exog = exog_all.reindex(train_ser.index).fillna(0)
    test_exog  = exog_all.reindex(test_ser.index).fillna(0)

    # Only include festival columns that actually appear in the training period
    active_cols = train_exog.columns[train_exog.sum() > 0].tolist()
    train_exog  = train_exog[active_cols] if active_cols else None
    test_exog_a = test_exog[active_cols]  if active_cols else None

    # Auto-detect if differencing is needed
    d_auto = get_d(train_ser)
    order  = (SARIMAX_ORDER[0], d_auto, SARIMAX_ORDER[2])

    try:
        model = SARIMAX(
            train_ser,
            exog                  = train_exog,
            order                 = order,
            seasonal_order        = SARIMAX_SEASONAL,
            enforce_stationarity  = False,
            enforce_invertibility = False,
        )
        result = model.fit(disp=False, maxiter=150)  # increased maxiter for better convergence

        # Forecast the test period
        preds = result.forecast(steps=len(test_ser), exog=test_exog_a)
        preds = preds.clip(lower=0)   # no negative predictions

        # Metrics
        mae   = mean_absolute_error(test_ser, preds)
        mape  = mean_absolute_percentage_error(test_ser, preds) * 100
        rmse  = float(np.sqrt(((test_ser.values - preds.values) ** 2).mean()))
        grade = 'A' if mape < 15 else ('B' if mape < 25 else 'C')

        # Extract festival coefficients from the fitted model
        coeffs = {}
        if active_cols:
            for col in active_cols:
                if col in result.params.index:
                    coeffs[col] = round(float(result.params[col]), 4)
        sarimax_coeffs[product] = coeffs

        # Save model
        safe = product.replace('/', '_').replace(' ', '_')[:40]
        with open(f'saved_models/sarimax/{safe}.pkl', 'wb') as fh:
            pickle.dump({
                'result':         result,
                'active_cols':    active_cols,
                'order':          order,
                'seasonal_order': SARIMAX_SEASONAL,
                'train_end':      train_ser.index.max()
            }, fh)

        sarimax_metrics.append({
            'Product':    product,
            'Train_Rows': len(train_ser),
            'Test_Rows':  len(test_ser),
            'd_used':     d_auto,
            'AIC':        round(result.aic, 2),
            'MAE':        round(mae, 2),
            'RMSE':       round(rmse, 2),
            'MAPE_%':     round(mape, 2),
            'Grade':      grade
        })
        print(f'  {product[:40]:<40} {mae:>8.1f} {mape:>7.1f}% {rmse:>8.1f} {d_auto:>4}  {grade}')

    except Exception as e:
        print(f'  {product[:40]:<40} FAILED: {e}')

# Save outputs
sarimax_metrics_df = pd.DataFrame(sarimax_metrics).sort_values('MAPE_%')
sarimax_metrics_df.to_csv('outputs/sarimax_metrics.csv', index=False)

sarimax_coeff_df = pd.DataFrame(sarimax_coeffs).T
sarimax_coeff_df.index.name = 'Product'
sarimax_coeff_df.to_csv('outputs/sarimax_festival_coefficients.csv')

print(f'\n✅ SARIMAX training complete: {len(sarimax_metrics)} models')
print(f'   Average MAPE: {sarimax_metrics_df["MAPE_%"].mean():.1f}%')
print(f'   Grade A (MAPE<15%): {(sarimax_metrics_df["Grade"]=="A").sum()} products')
sarimax_metrics_df

---
## 🌲 Cell 10 — Train XGBoost Model (Objective 2 — Improved)
**What is XGBoost?** A powerful gradient-boosting ML model that learns from features (columns) rather than the time-series structure directly.

**Why add XGBoost?**  
- Prophet and SARIMAX are excellent for time-series patterns  
- XGBoost can capture **non-linear interactions** between features (e.g., "Diwali on a Saturday boosts sales more than Diwali on a Tuesday")  
- Often gives **better accuracy** when combined in an ensemble

**Features we create (feature engineering):**
| Feature | What it captures |
|---|---|
| `day_of_week` | Mon=0 … Sun=6 |
| `month` | 1–12 (seasonal pattern) |
| `day_of_year` | 1–365 (yearly cycle) |
| `is_weekend` | 1 if Sat/Sun, else 0 |
| `is_festival` | 1 if any festival active, else 0 |
| `festival_name_*` | One-hot encoded festival identity |
| `lag_7`, `lag_14`, `lag_30` | Sales 7/14/30 days ago (memory) |
| `rolling_7_mean` | Average of last 7 days (smoothed trend) |

In [ ]:
def make_xgb_features(df_prod, exog_all, known_festivals):
    """
    Build feature matrix for XGBoost from a single product's daily time series.
    Returns: X (features), y (target quantity)
    """
    # Set daily frequency
    ts = df_prod.set_index('ds').asfreq('D').fillna(method='ffill').copy()

    # Calendar features
    ts['day_of_week']  = ts.index.dayofweek          # 0=Mon, 6=Sun
    ts['month']        = ts.index.month               # 1–12
    ts['day_of_year']  = ts.index.dayofyear           # 1–365
    ts['is_weekend']   = (ts.index.dayofweek >= 5).astype(int)
    ts['week_of_year'] = ts.index.isocalendar().week.astype(int)

    # Festival features (align exog matrix)
    exog_aligned = exog_all.reindex(ts.index).fillna(0)
    ts['is_festival']        = exog_aligned.max(axis=1)   # 1 if any festival active
    ts['num_festivals_today'] = exog_aligned.sum(axis=1)  # count of overlapping festivals
    for fest in known_festivals:
        if fest in exog_aligned.columns:
            ts[f'fest_{fest[:20]}'] = exog_aligned[fest]

    # Lag features (past sales as predictors)
    ts['lag_7']         = ts['y'].shift(7)    # same day last week
    ts['lag_14']        = ts['y'].shift(14)   # 2 weeks ago
    ts['lag_30']        = ts['y'].shift(30)   # last month
    ts['rolling_7_mean']  = ts['y'].shift(1).rolling(7).mean()   # 7-day moving avg
    ts['rolling_30_mean'] = ts['y'].shift(1).rolling(30).mean()  # 30-day moving avg

    ts = ts.dropna()   # remove rows where lag values are NaN
    feature_cols = [c for c in ts.columns if c != 'y']
    return ts[feature_cols], ts['y']


xgb_metrics = []
xgb_spikes  = {}

print(f'Training {len(products)} XGBoost models...\n')
print(f'{"Product":<40} {"MAE":>8} {"MAPE%":>8} {"RMSE":>8} {"Grade":>6}')
print('-' * 75)

for product in products:
    df_p = df[df['product'] == product][['ds', 'y']].dropna().copy()

    try:
        X, y = make_xgb_features(df_p, exog_all, known_festivals)

        # Train/test split by time
        cutoff   = y.index.max() - pd.Timedelta(days=TEST_DAYS)
        X_train, y_train = X[X.index <= cutoff], y[y.index <= cutoff]
        X_test,  y_test  = X[X.index  > cutoff], y[y.index  > cutoff]

        if len(X_train) < 30 or len(X_test) == 0:
            print(f'  {product[:40]:<40} SKIPPED (train={len(X_train)}, test={len(X_test)})')
            continue

        # Train XGBoost
        xgb = XGBRegressor(
            n_estimators      = 300,
            max_depth         = 4,
            learning_rate     = 0.05,
            subsample         = 0.8,
            colsample_bytree  = 0.8,
            random_state      = 42,
            verbosity         = 0
        )
        xgb.fit(X_train, y_train,
                eval_set=[(X_test, y_test)],
                early_stopping_rounds=30,
                verbose=False)

        preds = xgb.predict(X_test).clip(min=0)

        mae   = mean_absolute_error(y_test, preds)
        mape  = mean_absolute_percentage_error(y_test, preds) * 100
        rmse  = float(np.sqrt(((y_test.values - preds) ** 2).mean()))
        grade = 'A' if mape < 15 else ('B' if mape < 25 else 'C')

        # Festival spike from XGBoost: predict with/without festival flag
        spike_map = {}
        for fest in known_festivals:
            feat_col = f'fest_{fest[:20]}'
            if feat_col in X_train.columns:
                X_with    = X_test.copy(); X_with[feat_col]    = 1
                X_without = X_test.copy(); X_without[feat_col] = 0
                pred_with    = xgb.predict(X_with).mean()
                pred_without = xgb.predict(X_without).mean()
                if pred_without > 0:
                    spike_map[fest] = round((pred_with - pred_without) / pred_without * 100, 2)
        xgb_spikes[product] = spike_map

        # Save model
        safe = product.replace('/', '_').replace(' ', '_')[:40]
        with open(f'saved_models/xgboost/{safe}.pkl', 'wb') as fh:
            pickle.dump({'model': xgb, 'features': X_train.columns.tolist()}, fh)

        xgb_metrics.append({
            'Product': product, 'Train_Rows': len(X_train), 'Test_Rows': len(X_test),
            'MAE': round(mae, 2), 'RMSE': round(rmse, 2),
            'MAPE_%': round(mape, 2), 'Grade': grade
        })
        print(f'  {product[:40]:<40} {mae:>8.1f} {mape:>7.1f}% {rmse:>8.1f}  {grade}')

    except Exception as e:
        print(f'  {product[:40]:<40} FAILED: {e}')

xgb_metrics_df = pd.DataFrame(xgb_metrics).sort_values('MAPE_%')
xgb_metrics_df.to_csv('outputs/xgboost_metrics.csv', index=False)

xgb_spike_df = pd.DataFrame(xgb_spikes).T
xgb_spike_df.index.name = 'Product'
xgb_spike_df.to_csv('outputs/xgboost_festival_spikes.csv')

print(f'\n✅ XGBoost training complete: {len(xgb_metrics)} models')
print(f'   Average MAPE: {xgb_metrics_df["MAPE_%"].mean():.1f}%')
xgb_metrics_df

---
## 📊 Cell 11 — Compare All Three Models (Objective 4)
**Objective 4:** Decision support — which model to trust per product.

**Ensemble weighting:**  
Instead of picking just one model, we combine all three using **inverse-MAPE weighting**:  
- A model with MAPE=10% gets a higher weight than one with MAPE=30%  
- Final prediction = weighted average of all three models

This ensemble is usually more accurate than any single model.

In [ ]:
# ── Merge all three metrics tables ──
p_df = prophet_metrics_df[['Product','MAPE_%','MAE','RMSE','Grade']].rename(
    columns={'MAPE_%':'P_MAPE','MAE':'P_MAE','RMSE':'P_RMSE','Grade':'P_Grade'})
s_df = sarimax_metrics_df[['Product','MAPE_%','MAE','RMSE','Grade']].rename(
    columns={'MAPE_%':'S_MAPE','MAE':'S_MAE','RMSE':'S_RMSE','Grade':'S_Grade'})
x_df = xgb_metrics_df[['Product','MAPE_%','MAE','RMSE','Grade']].rename(
    columns={'MAPE_%':'X_MAPE','MAE':'X_MAE','RMSE':'X_RMSE','Grade':'X_Grade'})

comp = p_df.merge(s_df, on='Product', how='outer').merge(x_df, on='Product', how='outer')

# Determine winner per product
def pick_winner(row):
    scores = {'Prophet': row.get('P_MAPE', 9999),
              'SARIMAX': row.get('S_MAPE', 9999),
              'XGBoost': row.get('X_MAPE', 9999)}
    return min(scores, key=scores.get)

comp['Winner'] = comp.apply(pick_winner, axis=1)

# ── Inverse-MAPE ensemble weights for all three models ──
def inv_mape_weights_3(p, s, x):
    p, s, x = max(p or 9999, 0.01), max(s or 9999, 0.01), max(x or 9999, 0.01)
    inv = [1/p, 1/s, 1/x]
    total = sum(inv)
    return round(inv[0]/total, 3), round(inv[1]/total, 3), round(inv[2]/total, 3)

weights = comp.apply(
    lambda r: pd.Series(
        inv_mape_weights_3(r.get('P_MAPE'), r.get('S_MAPE'), r.get('X_MAPE')),
        index=['Prophet_W', 'SARIMAX_W', 'XGBoost_W']
    ), axis=1
)
comp = pd.concat([comp, weights], axis=1)

comp.to_csv('outputs/model_comparison.csv', index=False)

# ── Summary ──
p_wins = (comp['Winner'] == 'Prophet').sum()
s_wins = (comp['Winner'] == 'SARIMAX').sum()
x_wins = (comp['Winner'] == 'XGBoost').sum()

print('=' * 65)
print(f'  Products compared    : {len(comp)}')
print(f'  Prophet wins (MAPE)  : {p_wins}  ({p_wins/len(comp)*100:.0f}%)')
print(f'  SARIMAX wins (MAPE)  : {s_wins}  ({s_wins/len(comp)*100:.0f}%)')
print(f'  XGBoost wins (MAPE)  : {x_wins}  ({x_wins/len(comp)*100:.0f}%)')
print(f'  Avg MAPE — Prophet   : {comp["P_MAPE"].mean():.2f}%')
print(f'  Avg MAPE — SARIMAX   : {comp["S_MAPE"].mean():.2f}%')
print(f'  Avg MAPE — XGBoost   : {comp["X_MAPE"].mean():.2f}%')
print('=' * 65)

# ── Bar chart ──
fig, ax = plt.subplots(figsize=(8, 4))
model_avg_mape = {
    'Prophet': comp['P_MAPE'].mean(),
    'SARIMAX': comp['S_MAPE'].mean(),
    'XGBoost': comp['X_MAPE'].mean()
}
bars = ax.bar(model_avg_mape.keys(), model_avg_mape.values(),
              color=[COLORS['Prophet'], COLORS['SARIMAX'], COLORS['XGBoost']],
              edgecolor='white', width=0.5)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=11)
ax.set_title('Average MAPE by Model (lower = better)', fontsize=13, fontweight='bold')
ax.set_ylabel('MAPE (%)')
ax.set_ylim(0, max(model_avg_mape.values()) * 1.3)
plt.tight_layout()
plt.savefig('outputs/model_comparison_chart.png', bbox_inches='tight', dpi=100)
plt.show()

print('\nPer-product comparison:')
comp[['Product','P_MAPE','S_MAPE','X_MAPE','Winner','Prophet_W','SARIMAX_W','XGBoost_W']]

---
## 🎉 Cell 12 — Festival Spike Query Function (Objective 4)
**Objective 4:** Decision support — manager queries any festival and sees which products will spike.

**How it works:**
- **Seen festival** (e.g., Diwali in your data): uses exact learned effect from all three models
- **New/unseen festival** (e.g., a new event): uses average effect across all known festivals as a conservative estimate
- **Ensemble spike** = weighted average of Prophet, SARIMAX, and XGBoost spikes
- **Extra Units Estimate** = Baseline daily sales × Ensemble spike %

In [ ]:
def predict_festival_spike(festival_name: str, top_n: int = 10,
                           show_chart: bool = True):
    """
    Query: which products spike most for a given festival?

    Parameters:
    -----------
    festival_name : str
        Any festival name. If it was in your training data, exact effects are used.
        If it is a NEW festival not seen before, an average estimate is returned.
    top_n : int
        How many top products to display (default: 10)
    show_chart : bool
        Whether to show a bar chart (default: True)

    Returns:
    --------
    DataFrame with per-product spike estimates and extra unit predictions
    """

    baselines_local = pd.read_csv('outputs/product_baselines.csv', index_col=0)
    known           = json.load(open('outputs/known_festivals.json'))
    weights_df      = pd.read_csv('outputs/model_comparison.csv').set_index('Product')

    is_seen    = festival_name in known
    source_tag = 'Learned from training data' if is_seen else \
                 f'Estimated (avg of {len(known)} known festivals — conservative)'

    # ── Collect spikes from all three models ──
    p_spikes = pd.read_csv('outputs/prophet_festival_spikes.csv', index_col=0)
    s_coeffs = pd.read_csv('outputs/sarimax_festival_coefficients.csv', index_col=0)
    x_spikes = pd.read_csv('outputs/xgboost_festival_spikes.csv', index_col=0)

    all_products = sorted(set(p_spikes.index) | set(s_coeffs.index) | set(x_spikes.index))
    rows = []

    for product in all_products:
        bm = baselines_local.loc[product, 'baseline_mean'] \
             if product in baselines_local.index else None

        # Prophet spike %
        if product in p_spikes.index:
            if is_seen and festival_name in p_spikes.columns:
                p_pct = float(p_spikes.loc[product, festival_name])
            else:
                p_pct = float(p_spikes.loc[product].dropna().mean() or 0)
        else:
            p_pct = 0

        # SARIMAX spike % (convert coefficient → % using baseline)
        if product in s_coeffs.index and bm and bm > 0:
            if is_seen and festival_name in s_coeffs.columns:
                coeff = float(s_coeffs.loc[product, festival_name])
            else:
                coeff = float(s_coeffs.loc[product].dropna().mean() or 0)
            s_pct = round((coeff / bm) * 100, 2)
        else:
            s_pct = 0

        # XGBoost spike %
        if product in x_spikes.index:
            if is_seen and festival_name in x_spikes.columns:
                x_pct = float(x_spikes.loc[product, festival_name])
            else:
                x_pct = float(x_spikes.loc[product].dropna().mean() or 0)
        else:
            x_pct = 0

        # Ensemble spike using per-product model weights
        if product in weights_df.index:
            pw = weights_df.loc[product, 'Prophet_W']
            sw = weights_df.loc[product, 'SARIMAX_W']
            xw = weights_df.loc[product, 'XGBoost_W']
        else:
            pw, sw, xw = 1/3, 1/3, 1/3

        ensemble_pct = round(p_pct * pw + s_pct * sw + x_pct * xw, 2)
        extra_units  = round(bm * ensemble_pct / 100, 1) if bm else None

        rows.append({
            'Product':         product,
            'Baseline_Daily':  round(bm, 1) if bm else None,
            'Prophet_Spike_%': round(p_pct, 2),
            'SARIMAX_Spike_%': round(s_pct, 2),
            'XGBoost_Spike_%': round(x_pct, 2),
            'Ensemble_Spike_%': ensemble_pct,
            'Extra_Units_Est': extra_units,
            'Source':          source_tag
        })

    result = pd.DataFrame(rows).sort_values('Ensemble_Spike_%', ascending=False).head(top_n)

    # ── Print table ──
    print(f"\n{'='*80}")
    print(f"  Festival : '{festival_name}'")
    print(f"  Status   : {'✅ Seen (exact effects from training)' if is_seen else '⚡ New/Unseen (conservative estimate)'}")
    print(f"  Source   : {source_tag}")
    print(f"{'='*80}")
    print(f"  {'Product':<38} {'P%':>7} {'S%':>7} {'X%':>7} {'Ensemble%':>10} {'Extra Units':>12}")
    print('  ' + '-'*85)
    for _, row in result.iterrows():
        print(f"  {row['Product'][:38]:<38} "
              f"{row['Prophet_Spike_%']:>6.1f}% "
              f"{row['SARIMAX_Spike_%']:>6.1f}% "
              f"{row['XGBoost_Spike_%']:>6.1f}% "
              f"{row['Ensemble_Spike_%']:>9.1f}% "
              f"{str(row['Extra_Units_Est']):>12}")

    # ── Chart ──
    if show_chart and len(result) > 0:
        fig, ax = plt.subplots(figsize=(10, max(4, len(result) * 0.5)))
        colors = result['Ensemble_Spike_%'].apply(
            lambda x: '#2ecc71' if x > 20 else ('#f39c12' if x > 5 else '#e74c3c'))
        bars = ax.barh(result['Product'].str[:35], result['Ensemble_Spike_%'],
                       color=colors, edgecolor='white')
        ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
        ax.set_title(f"Demand Spike Prediction — {festival_name}\n"
                     f"({'Exact' if is_seen else 'Estimated'} | Ensemble of Prophet + SARIMAX + XGBoost)",
                     fontsize=12, fontweight='bold')
        ax.set_xlabel('Ensemble Spike %')
        ax.invert_yaxis()
        ax.axvline(0, color='black', linewidth=0.8)
        plt.tight_layout()
        plt.savefig(f'outputs/spike_{festival_name.replace(" ","_")}.png',
                    bbox_inches='tight', dpi=100)
        plt.show()

    return result.reset_index(drop=True)

print('✅ predict_festival_spike() is ready!')
print('\nUsage:')
print('  result = predict_festival_spike("Diwali")          # seen festival')
print('  result = predict_festival_spike("New Year", top_n=5)  # any festival')

---
## 🔍 Cell 13 — Run Festival Queries
Now query any festival to get production planning recommendations.

In [ ]:
# First, see all festivals available in your dataset
known_fests = json.load(open('outputs/known_festivals.json'))
print(f'Your dataset has {len(known_fests)} known festivals:\n')
for i, f in enumerate(known_fests, 1):
    print(f'  {i:2}. {f}')

print('\n' + '-'*50)
print('Uncomment one line below and run this cell:')
print('-'*50)

# ── ★ PICK YOUR FESTIVAL ──
# result = predict_festival_spike('Diwali')           # ← known festival
# result = predict_festival_spike('Gudi Padwa')       # ← known festival
# result = predict_festival_spike('Holi')             # ← known festival
# result = predict_festival_spike('Christmas', top_n=5)  # ← will estimate if unseen

# ── Run ALL known festivals and save a summary ──
# all_results = []
# for fest in known_fests:
#     r = predict_festival_spike(fest, show_chart=False)
#     r['Festival'] = fest
#     all_results.append(r)
# all_summary = pd.concat(all_results, ignore_index=True)
# all_summary.to_csv('outputs/all_festivals_summary.csv', index=False)
# print('\n✅ All festivals summary saved!')
# all_summary.head(20)

---
## 💾 Cell 14 — Save All Outputs
Downloads a zip of all outputs for your report and web dashboard.

In [ ]:
import shutil

# Create zip of all outputs
shutil.make_archive('dairy_model_outputs', 'zip', 'outputs')

# Auto-download in Colab
try:
    from google.colab import files
    files.download('dairy_model_outputs.zip')
except ImportError:
    print('Running locally — find your outputs in the outputs/ folder and dairy_model_outputs.zip')

print('\n📦 Contents of outputs/:')
print('  ── YEAR-WISE ANALYSIS (NEW) ──')
print('  yearwise_spike_analysis.csv       ← master table: Product × Festival × Year → Spike%')
print('  yearwise_next_year_forecast.csv   ← predicted spike & recommended qty for next year')
print('  yearwise_baselines.csv            ← per-year per-product normal day baseline')
print('  yearwise_heatmaps/                ← heatmap PNG per festival')
print('  yearwise_trends/                  ← trend+forecast chart PNG per festival')
print('  ── MODEL METRICS ──')
print('  prophet_metrics.csv              ← MAE/MAPE/RMSE/Grade per product (Prophet)')
print('  sarimax_metrics.csv              ← same for SARIMAX + AIC')
print('  xgboost_metrics.csv              ← same for XGBoost')
print('  model_comparison.csv             ← side-by-side + ensemble weights per product')
print('  prophet_festival_spikes.csv      ← % spike per festival per product')
print('  sarimax_festival_coefficients.csv← extra units per festival (SARIMAX coeffs)')
print('  xgboost_festival_spikes.csv      ← % spike from XGBoost counterfactual')
print('  product_baselines.csv            ← normal day mean/std per product')
print('  festival_demand_stats.csv        ← actual observed spike % per festival')
print('  festival_exog_matrix.csv         ← 0/1 festival calendar matrix')
print('  known_festivals.json             ← list of all festivals')
print('  seasonal_trends.png              ← monthly trend charts (Objective 1)')
print('  festival_vs_normal.png           ← festival vs normal day comparison')
print('  model_comparison_chart.png       ← MAPE comparison bar chart')
print('  spike_<festival>.png             ← spike charts per queried festival')
print('\nModel files saved in:')
print('  saved_models/prophet/            ← .pkl per product')
print('  saved_models/sarimax/            ← .pkl per product')
print('  saved_models/xgboost/            ← .pkl per product')